In [3]:
import openpyxl
import pandas as pd
import numpy as np
from IPython.core.display_functions import display
import openpyxl

# Load dataset (excel)
df = pd.read_excel("Country_data.xlsx")

In [4]:
# Inspect dataset
display(df.info())
display(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   country     167 non-null    str    
 1   child_mort  167 non-null    float64
 2   exports     167 non-null    float64
 3   health      167 non-null    float64
 4   imports     167 non-null    float64
 5   income      167 non-null    int64  
 6   inflation   167 non-null    float64
 7   life_expec  167 non-null    float64
 8   total_fer   167 non-null    float64
 9   gdpp        167 non-null    int64  
dtypes: float64(7), int64(2), str(1)
memory usage: 14.5 KB


None

,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
count,167.000000,167.000000,167.000000,167.000000,167.000000,167.000000,167.000000,167.000000,167.000000
mean,38.270060,41.108976,6.815689,46.890215,17144.688623,7.781832,70.555689,2.947964,12964.155689
std,40.328931,27.412010,2.746837,24.209589,19278.067698,10.570704,8.893172,1.513848,18328.704809
min,2.600000,0.109000,1.810000,0.065900,609.000000,-4.210000,32.100000,1.150000,231.000000
25%,8.250000,23.800000,4.920000,30.200000,3355.000000,1.810000,65.300000,1.795000,1330.000000
50%,19.300000,35.000000,6.320000,43.300000,9960.000000,5.390000,73.100000,2.410000,4660.000000
75%,62.100000,51.350000,8.600000,58.750000,22800.000000,10.750000,76.800000,3.880000,14050.000000
max,208.000000,200.000000,17.900000,174.000000,125000.000000,104.000000,82.800000,7.490000,105000.000000


In [5]:
# Check missing values
display(df.isnull().sum())

country       0
child_mort    0
exports       0
health        0
imports       0
income        0
inflation     0
life_expec    0
total_fer     0
gdpp          0
dtype: int64

In [6]:
# Drop duplicates
df.drop_duplicates(inplace=True)

In [7]:
# Ensure numeric columns
numeric_cols = ['child_mort','exports','health','imports','income','inflation','life_expec','total_fer','gdpp']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [8]:
display(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   country     167 non-null    str    
 1   child_mort  167 non-null    float64
 2   exports     167 non-null    float64
 3   health      167 non-null    float64
 4   imports     167 non-null    float64
 5   income      167 non-null    int64  
 6   inflation   167 non-null    float64
 7   life_expec  167 non-null    float64
 8   total_fer   167 non-null    float64
 9   gdpp        167 non-null    int64  
dtypes: float64(7), int64(2), str(1)
memory usage: 14.5 KB


None

In [9]:
# Detect & Outlier treatment using IQR
for col in ['income','gdpp','child_mort','inflation']:
    Q1, Q3 = df[col].quantile([0.25,0.75])
    display(Q1)
    display(Q3)
    IQR = Q3 - Q1
    df[col] = np.where(df[col] > Q3+1.5*IQR, Q3+1.5*IQR,
                       np.where(df[col] < Q1-1.5*IQR, Q1-1.5*IQR, df[col]))
    display(df[col])

3355.0

22800.0

0       1610.0
1       9930.0
2      12900.0
3       5900.0
4      19100.0
        ...   
162     2950.0
163    16500.0
164     4490.0
165     4480.0
166     3280.0
Name: income, Length: 167, dtype: float64

1330.0

14050.0

...

62.1

0       90.2
1       16.6
2       27.3
3      119.0
4       10.3
       ...  
162     29.2
163     17.1
164     23.3
165     56.3
166     83.1
Name: child_mort, Length: 167, dtype: float64

1.81

10.75

0       9.44
1       4.49
2      16.10
3      22.40
4       1.44
       ...  
162     2.62
163    24.16
164    12.10
165    23.60
166    14.00
Name: inflation, Length: 167, dtype: float64

In [10]:
# Replace negative inflation with NaN or cap at 0
df["inflation"] = df["inflation"].apply(lambda x: max(x, 0))

# Cap extreme inflation
df["inflation"] = np.where(df["inflation"] > 100, 100, df["inflation"])


In [11]:
display((df["income"] >= 0).all())       # Income should be non-negative
display((df["gdpp"] >= 0).all())         # GDP per capita should be non-negative
display((df["child_mort"] >= 0).all())   # Child mortality should be non-negative

np.True_

np.True_

np.True_

In [12]:
# Derived Features
df["Development_Index"] = (df["income"] + df["gdpp"] + df["life_expec"]) / df["child_mort"]

df["Trade_Balance"] = df["exports"] - df["imports"]

df["Health_Impact_Ratio"] = df["health"] / df["child_mort"]

In [13]:
# Define segmentation logic
def assign_segment(row):
    if row["child_mort"] > 80 and row["income"] < 5000:
        return "High Risk Country"
    elif row["income"] > 30000 and row["life_expec"] > 78:
        return "Developed Nation"
    elif 8000 <= row["income"] <= 30000:
        return "Emerging Economy"
    elif row["inflation"] > 15:
        return "High Inflation Risk"
    elif row["health"] < 5 and row["child_mort"] > 70:
        return "Health Critical"
    elif row["gdpp"] < 2000:
        return "Low GDP Trap"
    else:
        return "Other"

df['Segment'] = df.apply(assign_segment, axis=1)

In [14]:
# Risk Flag

#def risk_flag(row):
    # Countries are flagged if they fall into any high-risk categories
    #if (row["child_mort"] > 80 and row["income"] < 5000) \
   #    or (row["inflation"] > 15) \
     #  or (row["health"] < 5 and row["child_mort"] > 70) \
     #  or (row["gdpp"] < 2000):
     #   return 1   # At Risk
    #else:
      #  return 0   # Not At Risk

#df["Risk_Flag"] = df.apply(risk_flag, axis=1)
# Quick check
#print(df[['Segment','Risk_Flag']].head())


def risk_flag(row):
    # Define risk conditions
    if row['child_mort'] > 80 and row['income'] < 5000:
        return 1   # High Risk Country
    elif row['inflation'] > 15:
        return 1   # High Inflation Risk
    elif row['health'] < 5 and row['child_mort'] > 70:
        return 1   # Health Critical
    elif row['gdpp'] < 2000:
        return 1   # Low GDP Trap
    else:
        return 0   # Not risky

df['Risk_Flag'] = df.apply(risk_flag, axis=1)

# Quick check: count risky vs non-risky countries
print(df['Risk_Flag'].value_counts())

Risk_Flag
0    99
1    68
Name: count, dtype: int64


In [15]:
# Optional: see which countries are flagged
risky_countries = df[df['Risk_Flag'] == 1][['country','Segment','income','gdpp','child_mort','inflation']]
print(risky_countries.head())

        country              Segment   income     gdpp  child_mort  inflation
0   Afghanistan    High Risk Country   1610.0    553.0        90.2       9.44
2       Algeria     Emerging Economy  12900.0   4460.0        27.3      16.10
3        Angola  High Inflation Risk   5900.0   3530.0       119.0      22.40
5     Argentina     Emerging Economy  18700.0  10300.0        14.5      20.90
12   Bangladesh         Low GDP Trap   2440.0    758.0        49.4       7.14


In [16]:
df

,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp,Development_Index,Trade_Balance,Health_Impact_Ratio,Segment,Risk_Flag
0,Afghanistan,90.2,10.0,7.58,44.9,1610.0,9.44,56.2,5.82,553.0,24.603104,-34.9,0.084035,High Risk Country,1
1,Albania,16.6,28.0,6.55,48.6,9930.0,4.49,76.3,1.65,4090.0,849.174699,-20.6,0.394578,Emerging Economy,0
2,Algeria,27.3,38.4,4.17,31.4,12900.0,16.10,76.5,2.89,4460.0,638.699634,7.0,0.152747,Emerging Economy,1
3,Angola,119.0,62.3,2.85,42.9,5900.0,22.40,60.1,6.16,3530.0,79.748739,19.4,0.023950,High Inflation Risk,1
4,Antigua and Barbuda,10.3,45.5,6.03,58.9,19100.0,1.44,76.8,2.13,12200.0,3046.291262,-13.4,0.585437,Emerging Economy,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,Vanuatu,29.2,46.6,5.25,52.7,2950.0,2.62,63.0,3.50,2970.0,204.897260,-6.1,0.179795,Other,0
163,Venezuela,17.1,28.5,4.91,17.6,16500.0,24.16,75.4,2.47,13500.0,1758.795322,10.9,0.287135,Emerging Economy,1
164,Vietnam,23.3,72.0,6.84,80.2,4490.0,12.10,73.1,1.95,1310.0,252.064378,-8.2,0.293562,Low GDP Trap,1
165,Yemen,56.3,30.0,5.18,34.4,4480.0,23.60,67.5,4.67,1310.0,104.040853,-4.4,0.092007,High Inflation Risk,1


In [17]:
df.shape

(167, 15)

In [18]:
df.head()

,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp,Development_Index,Trade_Balance,Health_Impact_Ratio,Segment,Risk_Flag
0,Afghanistan,90.2,10.0,7.58,44.9,1610.0,9.44,56.2,5.82,553.0,24.603104,-34.9,0.084035,High Risk Country,1
1,Albania,16.6,28.0,6.55,48.6,9930.0,4.49,76.3,1.65,4090.0,849.174699,-20.6,0.394578,Emerging Economy,0
2,Algeria,27.3,38.4,4.17,31.4,12900.0,16.10,76.5,2.89,4460.0,638.699634,7.0,0.152747,Emerging Economy,1
3,Angola,119.0,62.3,2.85,42.9,5900.0,22.40,60.1,6.16,3530.0,79.748739,19.4,0.023950,High Inflation Risk,1
4,Antigua and Barbuda,10.3,45.5,6.03,58.9,19100.0,1.44,76.8,2.13,12200.0,3046.291262,-13.4,0.585437,Emerging Economy,0


In [19]:
# Save to CSV file
df.to_csv("cleaned_Country_data.csv", index=False)

print("✅ Cleaned dataset saved as 'cleaned_country_data.csv'")

✅ Cleaned dataset saved as 'cleaned_country_data.csv'


In [20]:
test = pd.read_csv("cleaned_Country_data.csv")
print(test.shape)

(167, 15)
